In [1]:
!pip install langchain langchain-community chromadb sentence-transformers -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 41.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 59.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 79.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 38.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 60.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.2/137.2 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 204.6/204.6 kB 12.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB

In [3]:
# 2. Build the Agricultural Knowledge Base
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document

knowledge = [
    Document(
        page_content="High humidity and continuous rain cause Late Blight in potato and tomato crops. Action: Ensure proper soil drainage and apply copper fungicide.",
        metadata={"topic": "Pest & Disease Management", "sdg": "SDG 2: Zero Hunger"}
    ),
    Document(
        page_content="Mulching and organic compost reduce soil evaporation by 35% during heat waves, preserving root temperature.",
        metadata={"topic": "Soil Health", "sdg": "SDG 13: Climate Action"}
    )
]

# 3. Create Vector Store (RAG)
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
vector_db = Chroma.from_documents(knowledge, embeddings)
retriever = vector_db.as_retriever(search_kwargs={"k": 1})
print("Knowledge base ready!")

/tmp/ipykernel_5405/165091624.py:18: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Knowledge base ready!


In [4]:
# 4. Agent Function
def agri_pulse_agent(query, sensor_data):
    docs = retriever.invoke(query)
    context = docs[0].page_content
    sdg = docs[0].metadata["sdg"]

    report = f"""
    =================== AGRIPULSE ADVISORY REPORT ===================
    Aligned UN Goal   : {sdg}
    Field Sensor Data : {sensor_data}
    Farmer Query      : {query}

    [Verified RAG Knowledge Grounding]:
    {context}

    [Recommended Agent Action Steps]:
    1. Immediate Action: Apply recommended mitigation to minimize crop loss.
    2. Input Optimization: Avoid excessive chemical spraying; target the exact issue.
    3. Climate Resilience: Maintain drainage and observe moisture changes.
    =================================================================
    """
    return report

In [5]:
# 5. Run Test
sample_sensors = {"Humidity": "86%", "Soil Moisture": "High", "Weather": "Heavy Rain"}
sample_query = "My tomato leaves are turning brown and soggy after the rain. What should I do?"

print(agri_pulse_agent(sample_query, sample_sensors))


    =================== AGRIPULSE ADVISORY REPORT ===================
    Aligned UN Goal   : SDG 2: Zero Hunger
    Field Sensor Data : {'Humidity': '86%', 'Soil Moisture': 'High', 'Weather': 'Heavy Rain'}
    Farmer Query      : My tomato leaves are turning brown and soggy after the rain. What should I do?
    
    [Verified RAG Knowledge Grounding]:
    High humidity and continuous rain cause Late Blight in potato and tomato crops. Action: Ensure proper soil drainage and apply copper fungicide.
    
    [Recommended Agent Action Steps]:
    1. Immediate Action: Apply recommended mitigation to minimize crop loss.
    2. Input Optimization: Avoid excessive chemical spraying; target the exact issue.
    3. Climate Resilience: Maintain drainage and observe moisture changes.
    
